In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
import random

import matplotlib.pyplot as plt
import pandas as pd
import pyspiel
from open_spiel.python.algorithms import outcome_sampling_mccfr
from tqdm.auto import trange

from cleptoninja import register_game as register_cleptoninja_game
from player import LowestOfferHighestBidIfOfferInHandPlayer, PolicyPlayer, RandomPlayer

In [ ]:
# TODO: Model players as a dataclass, hold their hands there. adapt rest of code
# TODO: Have auction resolution whithin dataclass. Query from game for card redistribution
# TODO: Change if phase ... for pattern matching everywhere

In [ ]:
register_cleptoninja_game()

In [ ]:
game = pyspiel.load_game("clepto_ninja")
solver = outcome_sampling_mccfr.OutcomeSamplingSolver(game)

for i in trange(100_000):
    solver.iteration()

avg_policy = solver.average_policy()

In [ ]:
def run_game(
    players=[RandomPlayer(id) for id in range(4)],
    debug=False,
):
    state = game.new_initial_state()
    while not state.is_terminal():
        player = state.current_player()
        action = players[player].action(state)
        state.apply_action(action)

        if debug:
            print(state)
            print()

    return state

In [ ]:
SIMULATION_COUNT = 100_000
ALL_MACHINE_PLAYERS = [
    LowestOfferHighestBidIfOfferInHandPlayer,
    PolicyPlayer,
    RandomPlayer,
]
player_count = 4

rows = []
for _ in range(SIMULATION_COUNT):
    player_classes = [random.choice(ALL_MACHINE_PLAYERS)] * player_count
    players = [
        (
            player_class(player_id)
            if player_class != PolicyPlayer
            else player_class(player_id, avg_policy)
        )
        for player_id, player_class in enumerate(player_classes)
    ]
    end_state = run_game(players)
    rows.append(end_state.returns() + [p.name for p in players])

game_results = pd.DataFrame(
    rows,
    columns=[f"payout_{i}" for i in range(player_count)]
    + [f"player_{i}" for i in range(player_count)],
)

In [ ]:
def plot_game_payouts(game_results, max_plotted_games=1_000):
    game_results = (
        game_results
        if len(game_results) < max_plotted_games
        else game_results.sample(max_plotted_games)
    )

    player_cols = [col for col in game_results if col.startswith("player_")]
    player_count = len(player_cols)
    _, ax = plt.subplots(figsize=(9, 3))

    # Build a stable color mapping for policies
    policies = pd.unique(game_results[player_cols].values.ravel())
    cmap = dict(zip(policies, plt.cm.tab10.colors[: len(policies)]))

    for i in range(player_count):
        ax.scatter(
            game_results.index,
            game_results[f"payout_{i}"],
            c=game_results[f"player_{i}"].map(cmap),
            s=20,
        )

    ax.set_xlabel("run")
    ax.set_ylabel("payout")
    ax.set_title(f"Player payouts ({SIMULATION_COUNT} simulations)")

    # Optional legend
    handles = [
        plt.Line2D([0], [0], marker="o", linestyle="", color=c, label=p)
        for p, c in cmap.items()
    ]
    ax.legend(
        handles=handles, title="player", bbox_to_anchor=(1.02, 1), loc="upper left"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
plot_game_payouts(game_results, max_plotted_games=1_000)

In [ ]:
def player_payout_stats(game_results):
    player_cols = [col for col in game_results if col.startswith("player_")]
    payout_cols = [col for col in game_results if col.startswith("payout_")]
    player_count = len(player_cols)

    long_df = pd.concat(
        [
            game_results[[payout_cols[i], player_cols[i]]].rename(
                columns={payout_cols[i]: "payout", player_cols[i]: "player"}
            )
            for i in range(player_count)
        ],
        ignore_index=True,
    )

    summary = (
        long_df.groupby("player")["payout"]
        .agg(
            min="min",
            q1=lambda x: x.quantile(0.25),
            median="median",
            mean="mean",
            q3=lambda x: x.quantile(0.75),
            max="max",
        )
        .reset_index()
    )

    return summary

In [ ]:
player_payout_stats(game_results)